# meshio Round-trip: FieldML -> VTU -> FieldML

`pyfieldml` includes a first-class two-way bridge to `meshio`, which unlocks every I/O format meshio supports (VTU, XDMF, Gmsh, Abaqus, Ansys, Nastran, ...). This notebook round-trips a dataset through `.vtu` and verifies that node coordinates survive unchanged.

In [ ]:
import tempfile
from pathlib import Path

import meshio
import numpy as np

import pyfieldml as fml
from pyfieldml import datasets

## Load a bundled dataset

In [ ]:
doc = datasets.load_unit_cube()
coords_before = doc.evaluators["coordinates"].as_ndarray()
print("nodes:", coords_before.shape)

## Convert to meshio and write as VTU

In [ ]:
m = doc.to_meshio()
print("meshio cells:", m.cells[0].type, "x", len(m.cells[0].data))

out_dir = Path(tempfile.mkdtemp(prefix="fieldml_rt_"))
vtu_path = out_dir / "unit_cube.vtu"
meshio.write(vtu_path, m)
print("wrote", vtu_path, f"({vtu_path.stat().st_size} bytes)")

## Read the VTU back with meshio and re-import into pyfieldml

In [ ]:
m2 = meshio.read(vtu_path)
doc2 = fml.Document.from_meshio(m2, name="roundtrip")
coords_after = doc2.evaluators["coordinates"].as_ndarray()
print("nodes after round-trip:", coords_after.shape)

## Verify node equality

In [ ]:
assert coords_before.shape == coords_after.shape, (
    f"shape mismatch: {coords_before.shape} vs {coords_after.shape}"
)
np.testing.assert_allclose(
    np.sort(coords_before, axis=0),
    np.sort(coords_after, axis=0),
    atol=1e-12,
)
print("OK: node coordinates survive the FieldML -> VTU -> FieldML round-trip.")

### What's lost, what's kept

**Kept**: node coordinates, element connectivity, per-node ParameterEvaluators (passed through `meshio.Mesh.point_data`).

**Lost**: the evaluator graph (basis external references, scale fields, derivative DOFs). meshio is a mesh-I/O layer and has no representation for FieldML's evaluator hierarchy, so exporting to VTU is lossy. Keep `.fieldml` for archival; use VTU/XDMF for visualization and FEM handoff.